### Install Dependencies

In [0]:
!pip install databricks-sdk -U
dbutils.library.restartPython()

### Define Parameters

In [0]:
dbutils.widgets.text("CATALOG_NAME", "workspace", "Catalog Name")
dbutils.widgets.text("SCHEMA_NAME", "retail_campaign", "Schema Name")
CATALOG_NAME = dbutils.widgets.get("CATALOG_NAME")
SCHEMA_NAME = dbutils.widgets.get("SCHEMA_NAME")


### Create Genie Space

UC Business Semantics metric views are natively integrated with AI/BI Genie, leveraging semantic metadata (instructions, synonyms, formatting) to deliver more accurate answers:  



Check out the sample serialized Genie space [./genie_space/genie_space.json](./genie_space/genie_space.json). 
You can use the code snippet below to automatically create it in your workspace, it uses the [Create Genie Space](https://docs.databricks.com/api/workspace/genie/createspace) API.  

In [0]:
import json
from databricks.sdk import WorkspaceClient

# Load the serialized Genie space from the JSON file
with open('./genie_space/genie_space.json', 'r') as f:
    genie_space_json_str = f.read()

# Replace CATALOG_NAME and SCHEMA_NAME placeholders with parameter values
catalog_name = dbutils.widgets.get("CATALOG_NAME")
schema_name = dbutils.widgets.get("SCHEMA_NAME")
genie_space_json_str = genie_space_json_str.replace("CATALOG_NAME", catalog_name).replace("SCHEMA_NAME", schema_name)

# Specify the warehouse ID (replace with your actual warehouse ID)
dbutils.widgets.text("warehouse_id", "", "Warehouse ID")
warehouse_id = dbutils.widgets.get("warehouse_id")

# Prepare the payload as required
payload = {
    "serialized_space": genie_space_json_str,
    "warehouse_id": warehouse_id,
    "title": "Sample Sales Genie Space (created via API)"
}

# Initialize WorkspaceClient (will use DATABRICKS_HOST and DATABRICKS_TOKEN from env)
w = WorkspaceClient()
workspace_url = w.config.host
pat_token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

import requests

headers = {
    "Authorization": f"Bearer {pat_token}",
    "Content-Type": "application/json"
}

# POST to Genie Create Space API
response = requests.post(
    f"{workspace_url}/api/2.0/genie/spaces",
    headers=headers,
    data=json.dumps(payload)
)

response.raise_for_status()
space_info = response.json()
print(f"Created Genie space with ID *{space_info['space_id']}* and name *{space_info['title']}*")

In [ ]:
dbutils.notebook.exit(json.dumps({"space_id": space_info["space_id"], "title": space_info["title"]}))